# Slot extractor — DeBERTa-v3 BIO token classification (Colab)
Self-contained. **Bật GPU runtime** (Runtime > Change runtime type > GPU).
Cell 1: upload `bio_train.jsonl`, `bio_val.jsonl`, `bio_test.jsonl` (từ `slot_extractor_model/data/`).
fp32 + `transformers==4.46.3` (5.x làm hỏng backbone DeBERTa-v3).

In [ ]:
# 1) Upload 3 file data BIO
from google.colab import files
up = files.upload()   # chọn bio_train.jsonl, bio_val.jsonl, bio_test.jsonl
print(sorted(up.keys()))

In [ ]:
# 2) Deps (pin transformers)
!pip -q install "transformers==4.46.3" datasets seqeval sentencepiece accelerate

In [ ]:
# 3) Train
import json, numpy as np
from datasets import Dataset
from seqeval.metrics import classification_report, f1_score
from transformers import (AutoModelForTokenClassification, AutoTokenizer,
    DataCollatorForTokenClassification, EarlyStoppingCallback, Trainer, TrainingArguments)

BACKBONE = "microsoft/deberta-v3-base"; MAX_LENGTH = 64
LABELS = ["O","B-COL","I-COL","B-FIT","I-FIT","B-OCC","I-OCC","B-SEA","I-SEA"]
L2I = {l:i for i,l in enumerate(LABELS)}; I2L = {i:l for l,i in L2I.items()}

def load(p):
    rows = [json.loads(l) for l in open(p, encoding="utf-8")]
    return Dataset.from_list([{"tokens": r["tokens"], "tags": [L2I[t] for t in r["tags"]]} for r in rows])
ds = {"train": load("bio_train.jsonl"), "val": load("bio_val.jsonl"), "test": load("bio_test.jsonl")}

tok = AutoTokenizer.from_pretrained(BACKBONE)
def align(b):
    enc = tok(b["tokens"], is_split_into_words=True, truncation=True, max_length=MAX_LENGTH)
    labs = []
    for i, tags in enumerate(b["tags"]):
        prev, seq = None, []
        for w in enc.word_ids(i):
            seq.append(-100 if w is None or w == prev else tags[w]); prev = w
        labs.append(seq)
    enc["labels"] = labs; return enc
ds = {k: v.map(align, batched=True, remove_columns=["tokens","tags"]) for k, v in ds.items()}

model = AutoModelForTokenClassification.from_pretrained(BACKBONE, num_labels=len(LABELS), id2label=I2L, label2id=L2I)
def metrics(p):
    pr = np.argmax(p.predictions, 2); tl, tp = [], []
    for a, l in zip(pr, p.label_ids):
        tl.append([I2L[x] for x in l if x != -100]); tp.append([I2L[y] for y, x in zip(a, l) if x != -100])
    return {"f1": f1_score(tl, tp)}

args = TrainingArguments(output_dir="ckpt", learning_rate=2e-5, num_train_epochs=12,
    per_device_train_batch_size=16, per_device_eval_batch_size=32, warmup_ratio=0.1,
    lr_scheduler_type="cosine", weight_decay=0.01, label_smoothing_factor=0.1,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="f1", greater_is_better=True, fp16=False, bf16=False,
    logging_steps=50, report_to="none", seed=42)
tr = Trainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["val"],
    data_collator=DataCollatorForTokenClassification(tok), compute_metrics=metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
tr.train()

In [ ]:
# 4) Test report + save
pred = tr.predict(ds["test"]); print("test f1:", pred.metrics.get("test_f1"))
pr = np.argmax(pred.predictions, 2); tl, tp = [], []
for a, l in zip(pr, pred.label_ids):
    tl.append([I2L[x] for x in l if x != -100]); tp.append([I2L[y] for y, x in zip(a, l) if x != -100])
print(classification_report(tl, tp))
tr.save_model("slot_extractor_deberta"); tok.save_pretrained("slot_extractor_deberta")

In [ ]:
# 5) Lưu model vào Google Drive (MyDrive/slot_extractor_deberta)
from google.colab import drive
drive.mount("/content/drive")
import shutil, os
dst = "/content/drive/MyDrive/slot_extractor_deberta"
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree("slot_extractor_deberta", dst)
print("saved ->", dst, "| files:", os.listdir(dst))